# 41 — One model, five views (high-fidelity neurons)

Same three **polyglot-complete** catalogue neurons through:
(1) membrane dynamics, (2) f–I curves, (3) SC rate approximation,
(4) simple quantised voltage view, (5) hardware-path pointer.

## Honesty box

| | |
|---|---|
| **Proves** | Runnable traces and f–I for Hodgkin–Huxley, Morris–Lecar, and AdEx using the package catalogue implementations; SC *scalar rate* approximation error vs bitstream length; a didactic fixed-point quantisation of a voltage sample. |
| **Does not prove** | Full polyglot parity in this notebook, FPGA board timing/power, formal equivalence, RTL co-simulation success for all three models, or Studio UI integration. |
| **Artefacts** | Local figures only unless you commit them. Prefer naming any promoted PNG under `benchmarks/` / docs with a separate ticket. |
| **Models** | `HodgkinHuxleyNeuron`, `MorrisLecarNeuron`, `AdExNeuron` — must remain on `docs/api/model_fidelity_status.md` polyglot-complete table at the checkout you use. |
| **Language** | British English. |


In [ ]:
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

from sc_neurocore.neurons.models import (
    AdExNeuron,
    HodgkinHuxleyNeuron,
    MorrisLecarNeuron,
)

np.random.seed(42)
print("SC-NeuroCore — NB-41 five views on high-fidelity neurons")


## 0. Confirm models resolve


In [ ]:
MODELS = {
    "HodgkinHuxleyNeuron": HodgkinHuxleyNeuron,
    "MorrisLecarNeuron": MorrisLecarNeuron,
    "AdExNeuron": AdExNeuron,
}
for name, cls in MODELS.items():
    n = cls()
    print(f"{name}: dt={getattr(n, 'dt', None)} simulate={callable(getattr(n, 'simulate', None))}")


## 1. Membrane dynamics (view 1)

Constant-current drive long enough to show spikes. Currents are chosen
from each model's enrolled operating envelope style (not universal units).


In [ ]:
@dataclass(frozen=True)
class Drive:
    n_steps: int
    current: float

DRIVES = {
    "HodgkinHuxleyNeuron": Drive(8000, 10.0),
    "MorrisLecarNeuron": Drive(4000, 50.0),
    "AdExNeuron": Drive(4000, 250.0),
}

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=False)
for ax, (name, cls) in zip(axes, MODELS.items()):
    drive = DRIVES[name]
    neuron = cls()
    v, spikes = neuron.simulate(drive.n_steps, current=drive.current)
    t = np.arange(len(v)) * float(neuron.dt)
    ax.plot(t, v, lw=0.9)
    ax.set_title(f"{name} — I={drive.current}, spikes={spikes}")
    ax.set_ylabel("v")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("time (model dt units)")
fig.tight_layout()
plt.show()


## 2. f–I curves (view 2)


In [ ]:
def fi_curve(cls, currents: np.ndarray, n_steps: int) -> np.ndarray:
    rates = []
    for i in currents:
        neuron = cls()
        _v, spikes = neuron.simulate(int(n_steps), current=float(i))
        duration_s = (n_steps * float(neuron.dt)) / 1000.0  # dt treated as ms
        rates.append(spikes / duration_s if duration_s > 0 else 0.0)
    return np.asarray(rates, dtype=float)

FI_SPECS = {
    "HodgkinHuxleyNeuron": (np.linspace(0, 20, 11), 5000),
    "MorrisLecarNeuron": (np.linspace(0, 100, 11), 3000),
    "AdExNeuron": (np.linspace(0, 500, 11), 3000),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, cls) in zip(axes, MODELS.items()):
    currents, n_steps = FI_SPECS[name]
    rates = fi_curve(cls, currents, n_steps)
    ax.plot(currents, rates, "o-", lw=1.2)
    ax.set_title(name.replace("Neuron", ""))
    ax.set_xlabel("I (model units)")
    ax.set_ylabel("rate (Hz, dt-as-ms)")
    ax.grid(True, alpha=0.3)
fig.suptitle("f–I curves (high-fidelity catalogue models)")
fig.tight_layout()
plt.show()


## 3. Stochastic computing rate approximation (view 3)

Encode a normalised firing proxy as a unipolar bitstream and measure
reconstruction error vs length. This is **SC arithmetic pedagogy**, not
a claim that the full HH/AdEx dynamics run as bitstreams here.


In [ ]:
from sc_neurocore import BitstreamEncoder, bitstream_to_probability

def sc_rate_error(true_rate_norm: float, lengths: list[int], seed: int = 0) -> list[float]:
    errs = []
    x = float(np.clip(true_rate_norm, 0.0, 1.0))
    for length in lengths:
        enc = BitstreamEncoder(x_min=0.0, x_max=1.0, length=length, seed=seed + length)
        bits = enc.encode(x)
        est = float(bitstream_to_probability(bits))
        errs.append(abs(est - x))
    return errs

# Use mid-drive rates from HH f–I as a unitless proxy in [0,1]
hh_currents, hh_steps = FI_SPECS["HodgkinHuxleyNeuron"]
hh_rates = fi_curve(HodgkinHuxleyNeuron, hh_currents, hh_steps)
proxy = float(hh_rates[len(hh_rates) // 2] / max(hh_rates.max(), 1.0))
lengths = [64, 128, 256, 512, 1024, 2048, 4096]
errs = sc_rate_error(proxy, lengths)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.loglog(lengths, errs, "o-", label=f"proxy={proxy:.3f}")
ax.set_xlabel("bitstream length")
ax.set_ylabel("|decode − value|")
ax.set_title("SC unipolar reconstruction error (pedagogy)")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
print("proxy value", proxy, "errors", list(zip(lengths, np.round(errs, 4))))


## 4. Didactic quantisation of a voltage trace (view 4)

Map a voltage window to Q8.8-style integers for illustration. This is
**not** the full package QAT/RTL pipeline (see notebook 13 / 08 / 27).


In [ ]:
def quantise_q88(v: np.ndarray, v_min: float, v_max: float) -> tuple[np.ndarray, np.ndarray]:
    scale = (v_max - v_min) / 256.0 if v_max > v_min else 1.0
    q = np.clip(np.rint((v - v_min) / scale), 0, 255).astype(np.int16)
    recon = q.astype(np.float64) * scale + v_min
    return q, recon

neuron = HodgkinHuxleyNeuron()
v, _ = neuron.simulate(4000, current=10.0)
q, recon = quantise_q88(v, float(v.min()), float(v.max()))
mse = float(np.mean((v - recon) ** 2))

fig, ax = plt.subplots(figsize=(10, 3))
t = np.arange(len(v)) * float(neuron.dt)
ax.plot(t, v, label="float", lw=1.0)
ax.plot(t, recon, label="Q8.8-style recon", lw=0.9, alpha=0.85)
ax.set_title(f"Hodgkin–Huxley voltage quantisation demo (MSE={mse:.4f})")
ax.set_xlabel("time")
ax.set_ylabel("v")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 5. Hardware-path pointer (view 5)

SC-NeuroCore already ships equation→Verilog / golden-path notebooks.
This cell only **records** next hops — it does not run synthesis.


In [ ]:
HARDWARE_NEXT = {
    "equation_to_verilog": "notebooks/08_equation_to_verilog.ipynb",
    "python_to_proven_silicon": "notebooks/27_python_to_proven_silicon.ipynb",
    "golden_path_evidence": "notebooks/29_golden_path_evidence.ipynb",
    "fidelity_table": "docs/api/model_fidelity_status.md",
}
print(json.dumps(HARDWARE_NEXT, indent=2))
print("NB-41 complete: dynamics + f–I + SC pedagogy + quant demo + hardware pointers.")
